# TCC Drone · Etapa 1.9.8 — NOVO VOO | pipeline integrado

**Objetivo:** processar o `IMG_5962_editado.mov` inteiro com seleção YOLO congelada (ou inferência com checkpoint congelado), regressão por largura e Depth Pro opcional, registrando abstenções e renderizando vídeo.

**Importante:** a regressão foi calibrada no FAROL e é uma **transferência exploratória**. Depth Pro fornece **Z axial**, não alcance radial certificado. Este notebook não executa treinamento, não usa GT para escolher detecção, não computa MAE físico sem referência independente e não transfere pesos automaticamente.

Os arquivos pequenos de seleção YOLO v1.9.5 já estão no ZIP do módulo. Os originais grandes (vídeo, `best.pt`, `depth_pro.pt`) devem ser colocados por você no Drive. Este notebook usa **o mesmo código Python do VS Code**, evitando duas implementações divergentes.

In [ ]:
# 1. Montar Drive
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, subprocess, zipfile, shutil, json

# AJUSTE ESTES CAMINHOS (Drive); nenhum arquivo será movido ou substituído sem aviso.
ZIP_MODULO = Path('/content/drive/MyDrive/TCC_Drone/MODULO_PIPELINE_INTEGRADO_NOVO_VOO_v1_9_8.zip')
VIDEO = Path('/content/drive/MyDrive/TCC_Drone/IMG_5962_editado.mov')
DEPTH_WEIGHTS = Path('/content/drive/MyDrive/TCC_Drone/depth_pro.pt')
RESULTADOS = Path('/content/drive/MyDrive/TCC_Drone/resultados_pipeline_v1_9_8')

# Proteção contra paths inexistentes
for caminho in (ZIP_MODULO, VIDEO):
    if not caminho.is_file(): raise FileNotFoundError(f'Edite o caminho e faça upload para o Drive: {caminho}')
RESULTADOS.mkdir(exist_ok=True,parents=True)


In [ ]:
# 2. Instalar dependências pequenas, sem alterar modelo e pesos
import importlib.util
if importlib.util.find_spec('cv2') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','opencv-python-headless'], check=True)
if importlib.util.find_spec('pandas') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','pandas'], check=True)
if importlib.util.find_spec('PIL') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','Pillow'], check=True)

# Extrair módulo para /content (não alterar fonte no Drive)
BASE = Path('/content/tcc_pipeline_v1_9_8')
BASE.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(ZIP_MODULO) as z:
    z.extractall(BASE)
SCRIPT = BASE/'ETAPA_PIPELINE_INTEGRADO_NOVO_VOO_v1_9_8'/'pipeline_integrado.py'
assert SCRIPT.is_file(), SCRIPT
print('Código:', SCRIPT)

def execute(stage, out_name, *flags):
    cmd=[sys.executable, '-u', str(SCRIPT),stage,
         '--video',str(VIDEO),'--saida',str(RESULTADOS/out_name),*flags]
    print('Executando:', ' '.join(cmd),flush=True)
    subprocess.run(cmd,check=True)


In [ ]:
# 3. Auditoria e vídeo piloto (4 s). O SHA256 do vídeo é sempre exigido no modo cache.
execute('auditar','auditoria')
execute('piloto','PILOTO_REGRESSAO','--max-frames','120')
print('Confira o vídeo:', RESULTADOS/'PILOTO_REGRESSAO'/'VIDEO_FINAL_EXPLORATORIO_SEM_AUDIO.mp4')


In [ ]:
# 4. Vídeo integral: 5.142 quadros; YOLO do arquivo congelado + regressão.
# NÃO há novas inferências YOLO; usa exatamente as caixas arquivadas.
# OBSERVE: valores transferidos do FAROL permanecem rotulados como EXPLORATÓRIOS.
execute('executar','INTEGRAL_REGRESSAO')
print('Saída integral:',RESULTADOS/'INTEGRAL_REGRESSAO'/'VIDEO_FINAL_EXPLORATORIO_SEM_AUDIO.mp4')


## 5. Depth Pro (opcional, mas necessário para completar os DOIS métodos)

O modelo Depth Pro deve ser instalado pelo seu repositório oficial; os pesos **não vêm neste pacote**. Este código extrai profundidade em quadro completo a cada 30 frames **selecionados** (≈1 Hz nos quadros com caixa confiável). Não cria profundidades para quadros não processados. Para fazer todos os quadros elegíveis, use `--depth-every 1`, ciente do grande custo de GPU/tempo. A execução pode ser retomada a partir dos resultados de Depth salvos.

**O vídeo continua tendo 5.142 quadros, mas não apresenta Z para todos eles.** A regressão usa todos os segundos com caixas suficientes. Protocolo de referência sem calibração física certificada.

In [ ]:
# 5a. Preparar Depth Pro e verificar checkpoint (execute uma vez se desejar profundidade)
# Fonte oficial: apple/ml-depth-pro. Instale somente se necessário.
DEPTH_REPO_ROOT = Path('/content/ml-depth-pro')
if not DEPTH_WEIGHTS.is_file():
    print('Pesos Depth Pro NÃO localizados:', DEPTH_WEIGHTS)
    print('Faça upload do depth_pro.pt ao Drive e corrija DEPTH_WEIGHTS antes de rodar a célula 5b.')
else:
    if not (DEPTH_REPO_ROOT/'src'/'depth_pro').is_dir():
        subprocess.run(['git','clone','--depth','1','https://github.com/apple/ml-depth-pro.git',str(DEPTH_REPO_ROOT)],check=True)
    # Para reprodutibilidade, consulte o commit impresso (o checkpoint deve ser compatível).
    subprocess.run(['git','-C',str(DEPTH_REPO_ROOT),'rev-parse','HEAD'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(DEPTH_REPO_ROOT)],check=True)
    print('Depth Pro instalado. Verifique GPU e dependências antes da célula seguinte.')


In [ ]:
# 5b. Executar Depth Pro em quadro inteiro, amostragem EXPLÍCITA 1 Hz
# Na GPU T4, memória e tempo dependem da implementação, 4K pode causar OOM.
if not DEPTH_WEIGHTS.is_file():
    raise FileNotFoundError('Para concluir com Depth Pro, informe DEPTH_WEIGHTS corretamente.')
execute('executar','INTEGRAL_COM_DEPTH_1HZ',
        '--depth-every','30','--depth-weights',str(DEPTH_WEIGHTS),
        '--depth-repo',str(DEPTH_REPO_ROOT/'src'))
print('Vídeo:',RESULTADOS/'INTEGRAL_COM_DEPTH_1HZ'/'VIDEO_FINAL_EXPLORATORIO_SEM_AUDIO.mp4')


In [ ]:
# 6. Ler o relatório; compacte apenas arquivos pequenos para auditoria
from IPython.display import display, Markdown
folder=RESULTADOS/'INTEGRAL_REGRESSAO'
display(Markdown((folder/'RELATORIO_INTEGRACAO_v1_9_8.md').read_text(encoding='utf-8')))
for name in ['PROVENIENCIA_v1_9_8.json','RESUMO_EXECUCAO_v1_9_8.json']:
    print(name, (folder/name).stat().st_size,'bytes')
